[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/07-cli-tools.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/07-cli-tools.ipynb)

# Module 3.7 — CLI Tools
**Module 3: Automation & Scripting** | Estimated time: 35 minutes

---

## Learning Objectives
By the end of this notebook you will be able to:
- Build argument parsers with `argparse` (positional args, options, flags, choices, subparsers)
- Create expressive CLI commands with `click` decorators
- Produce beautiful terminal output with `rich` (tables, progress bars, panels, syntax highlighting)
- Understand how to package a CLI script for distribution
- Write the same tool in both `argparse` and `click` to understand the trade-offs

In [ ]:
!pip install click rich -q

import argparse
import sys
import os
import json
from pathlib import Path

import click
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import track, Progress, SpinnerColumn, BarColumn, TextColumn
from rich.syntax import Syntax
from rich.text import Text
from rich import print as rprint

import time
import click

print('click version:', click.__version__)
print('rich  version:', __import__('rich').__version__)

## 1. `argparse` — Built-in Argument Parsing

`argparse` is the standard library solution. It automatically generates `--help` text and validates types.

In [ ]:
# Build a parser for a hypothetical file converter tool
parser = argparse.ArgumentParser(
    prog='convert',
    description='Convert data files between formats.',
    formatter_class=argparse.RawDescriptionHelpFormatter,
    epilog='Examples:\n  convert data.csv output.json\n  convert report.json report.yaml --indent 4',
)

# Positional arguments
parser.add_argument('input',  help='Input file path')
parser.add_argument('output', help='Output file path')

# Optional arguments
parser.add_argument('-i', '--indent',   type=int, default=2,
                    help='JSON indent level (default: 2)')
parser.add_argument('-e', '--encoding', default='utf-8',
                    help='File encoding (default: utf-8)')
parser.add_argument('--overwrite', action='store_true',
                    help='Overwrite output file if it exists')
parser.add_argument('--format', choices=['json', 'yaml', 'csv', 'tsv'],
                    help='Force output format (inferred from extension if omitted)')
parser.add_argument('-v', '--verbose', action='count', default=0,
                    help='Verbosity: -v info, -vv debug')

# Print the auto-generated help text
parser.print_help()

In [ ]:
# Simulate parsing different argument strings
test_cases = [
    ['data.csv', 'output.json'],
    ['data.csv', 'output.json', '--indent', '4', '--overwrite'],
    ['report.json', 'report.yaml', '-v', '-v', '--format', 'yaml'],
]

for args_list in test_cases:
    ns = parser.parse_args(args_list)
    print(f'args: {args_list}')
    print(f'  input={ns.input!r}, output={ns.output!r}, indent={ns.indent},'
          f' overwrite={ns.overwrite}, format={ns.format!r}, verbose={ns.verbose}')
    print()

## 2. `argparse` Subparsers

Subparsers let you create git-style commands (`tool commit`, `tool push`, etc.).

In [ ]:
main_parser = argparse.ArgumentParser(prog='pypath-tool')
main_parser.add_argument('--config', default='~/.pypath.json', help='Config file path')

subs = main_parser.add_subparsers(dest='command', title='commands')

# 'run' subcommand
run_p = subs.add_parser('run', help='Run a scraping job')
run_p.add_argument('job',            help='Job name or path to job file')
run_p.add_argument('--dry-run',      action='store_true', help='Preview without executing')
run_p.add_argument('--max-pages',    type=int, default=10, metavar='N')

# 'export' subcommand
export_p = subs.add_parser('export', help='Export collected data')
export_p.add_argument('dest',   help='Destination path')
export_p.add_argument('--fmt',  choices=['json', 'csv', 'xlsx'], default='json')
export_p.add_argument('--since', metavar='YYYY-MM-DD', help='Filter records since date')

# 'status' subcommand
subs.add_parser('status', help='Show scheduler status')

# Demo: parse a 'run' command
ns = main_parser.parse_args(['--config', '/etc/pypath.json', 'run', 'books_job', '--max-pages', '5'])
print('Command  :', ns.command)
print('Config   :', ns.config)
print('Job      :', ns.job)
print('Dry run  :', ns.dry_run)
print('Max pages:', ns.max_pages)

## 3. `click` — Decorator-Based CLI Framework

`click` uses decorators to define commands, which produces cleaner, more Pythonic code.

In [ ]:
# In a real script you would use `if __name__ == '__main__': cli()`
# In a notebook we call cli.main(args, standalone_mode=False)

@click.group()
@click.option('--config', default='~/.pypath.json', show_default=True, help='Config file')
@click.pass_context
def cli(ctx, config):
    """PyPath automation toolkit."""
    ctx.ensure_object(dict)
    ctx.obj['config'] = config


@cli.command()
@click.argument('input_file', type=click.Path(exists=False))
@click.argument('output_file', type=click.Path())
@click.option('-i', '--indent', default=2, show_default=True, type=int, help='JSON indent')
@click.option('--overwrite', is_flag=True, help='Overwrite if output exists')
@click.option('--fmt', type=click.Choice(['json', 'yaml', 'csv']), help='Output format')
@click.pass_context
def convert(ctx, input_file, output_file, indent, overwrite, fmt):
    """Convert a data file between formats."""
    click.echo(f'Config   : {ctx.obj["config"]}')
    click.echo(f'Input    : {input_file}')
    click.echo(f'Output   : {output_file}')
    click.echo(f'Indent   : {indent}')
    click.echo(f'Overwrite: {overwrite}')
    click.echo(f'Format   : {fmt}')


@cli.command()
@click.option('--name', prompt='Your name', help='Greeted user')
@click.option('--count', default=1, show_default=True, type=int, help='Greet N times')
def greet(name, count):
    """Greet a user."""
    for _ in range(count):
        click.echo(click.style(f'Hello, {name}!', fg='green', bold=True))


# Invoke convert without prompts
result = cli.main(
    ['--config', '/etc/pypath.json', 'convert', 'data.csv', 'data.json', '--indent', '4'],
    standalone_mode=False
)
print()

# Invoke greet
result = cli.main(['greet', '--name', 'Pythonista', '--count', '3'], standalone_mode=False)

## 4. `click` — Passwords, File Params, and Confirmation

In [ ]:
@click.command()
@click.option('--username', prompt=True, help='API username')
@click.option('--password', prompt=True, hide_input=True, confirmation_prompt=True,
              help='API password')
@click.option('--output', type=click.File('w'), default='-', help='Output file (- for stdout)')
def login_demo(username, password, output):
    """Demonstrate secure prompts (password is masked)."""
    output.write(f'User: {username}\n')
    output.write(f'Password hash: {hash(password)}\n')  # never log real passwords!

# Simulate non-interactive call
login_demo.main(
    ['--username', 'alice', '--password', 'secret'],
    standalone_mode=False
)

## 5. `rich` — Beautiful Terminal Output

In [ ]:
console = Console()

# Markup: bold, italic, colors, links
console.print('[bold blue]PyPath[/bold blue] [italic]Automation Toolkit[/italic] v1.0')
console.print('[green]SUCCESS[/green] All jobs completed.')
console.print('[red bold]ERROR[/red bold] Connection refused.')
console.print('[yellow]WARNING[/yellow] Rate limit approaching (80%).')

# Panel
console.print(Panel(
    '[bold]Module 3: Automation & Scripting[/bold]\n'
    'Building powerful automation scripts with Python.',
    title='[blue]PyPath[/blue]',
    border_style='blue',
    padding=(1, 2)
))

In [ ]:
# Rich Table
table = Table(title='Web Scraping Jobs', show_lines=True)
table.add_column('Job',        style='cyan',  no_wrap=True)
table.add_column('Status',     style='bold')
table.add_column('Pages',      justify='right')
table.add_column('Items',      justify='right')
table.add_column('Last Run',   style='dim')

rows = [
    ('books_catalogue',  '[green]done[/green]',    '50', '1000', '2024-03-15 09:02'),
    ('product_prices',   '[green]done[/green]',    '12',  '240', '2024-03-15 09:15'),
    ('news_headlines',   '[yellow]running[/yellow]','--',   '--', '2024-03-15 09:30'),
    ('competitor_data',  '[red]failed[/red]',       '3',   '56', '2024-03-14 22:00'),
    ('stock_quotes',     '[dim]pending[/dim]',      '--',   '--', 'never'),
]
for row in rows:
    table.add_row(*row)

console.print(table)

In [ ]:
# Rich Progress bar
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeRemainingColumn

urls = [f'https://example.com/page/{i}' for i in range(1, 11)]

with Progress(
    SpinnerColumn(),
    TextColumn('[progress.description]{task.description}'),
    BarColumn(),
    TextColumn('[progress.percentage]{task.percentage:>3.0f}%'),
    TimeRemainingColumn(),
    console=console,
) as progress:
    task = progress.add_task('[cyan]Scraping pages...', total=len(urls))
    results = []
    for url in urls:
        time.sleep(0.1)  # simulate work
        results.append({'url': url, 'status': 200})
        progress.advance(task)

console.print(f'[bold green]Done![/bold green] Scraped {len(results)} pages.')

In [ ]:
# Rich Syntax highlighting
code = '''
def scrape_page(url: str, session: requests.Session) -> list[dict]:
    """Fetch a page and extract product listings."""
    resp = session.get(url, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    return [
        {"title": el.h3.text, "price": el.select_one(".price").text}
        for el in soup.select("article.product")
    ]
'''

syntax = Syntax(code, 'python', theme='monokai', line_numbers=True,
                highlight_lines={3, 4})
console.print(Panel(syntax, title='[bold]Highlighted Code[/bold]', border_style='magenta'))

## 6. Complete CLI Tool — `argparse` vs `click` Versions

Both versions implement the same `file-stats` tool: scan a directory and report file counts and sizes by extension.

In [ ]:
# ── Shared business logic ──────────────────────────────────────────────────
from collections import defaultdict

def analyze_directory(path: str, recursive: bool = False) -> dict:
    """Return {extension: {count, total_bytes}} for every file under path."""
    root = Path(path)
    if not root.is_dir():
        raise ValueError(f'{path!r} is not a directory')
    glob_fn = root.rglob if recursive else root.glob
    stats = defaultdict(lambda: {'count': 0, 'bytes': 0})
    for f in glob_fn('*'):
        if f.is_file():
            ext = f.suffix.lower() or '(none)'
            stats[ext]['count'] += 1
            stats[ext]['bytes'] += f.stat().st_size
    return dict(stats)


def format_size(n: int) -> str:
    for unit in ('B', 'KB', 'MB', 'GB'):
        if n < 1024:
            return f'{n:.1f} {unit}'
        n /= 1024
    return f'{n:.1f} TB'


# ── argparse version ──────────────────────────────────────────────────────
def file_stats_argparse(argv=None):
    p = argparse.ArgumentParser(prog='file-stats', description='Directory file statistics')
    p.add_argument('path', help='Directory to scan')
    p.add_argument('-r', '--recursive', action='store_true')
    p.add_argument('--sort', choices=['count', 'size', 'ext'], default='size')
    p.add_argument('--top', type=int, default=10, metavar='N')
    args = p.parse_args(argv or [])
    stats = analyze_directory(args.path, args.recursive)
    rows = sorted(stats.items(), key=lambda x: -x[1][args.sort if args.sort != 'size' else 'bytes'])
    print(f'\n{"Extension":15s} {"Count":>8s} {"Total Size":>12s}')
    print('-' * 37)
    for ext, s in rows[:args.top]:
        print(f'{ext:15s} {s["count"]:>8d} {format_size(s["bytes"]):>12s}')


# ── click version ─────────────────────────────────────────────────────────
@click.command('file-stats')
@click.argument('path', default='.')
@click.option('-r', '--recursive', is_flag=True, help='Recurse into subdirectories')
@click.option('--sort', type=click.Choice(['count', 'size', 'ext']), default='size',
              show_default=True, help='Sort column')
@click.option('--top', default=10, show_default=True, metavar='N', help='Show top N extensions')
def file_stats_click(path, recursive, sort, top):
    """Show file count and size grouped by extension."""
    stats = analyze_directory(path, recursive)
    key = 'bytes' if sort == 'size' else sort
    rows = sorted(stats.items(), key=lambda x: (-x[1][key] if sort != 'ext' else x[0]))

    con = Console()
    tbl = Table(title=f'File Stats: {path}', show_lines=False)
    tbl.add_column('Extension', style='cyan')
    tbl.add_column('Count', justify='right')
    tbl.add_column('Total Size', justify='right', style='green')
    for ext, s in rows[:top]:
        tbl.add_row(ext, str(s['count']), format_size(s['bytes']))
    con.print(tbl)


# Run both versions on /tmp
print('=== argparse version ===')
file_stats_argparse(['/tmp', '--recursive', '--top', '5'])
print()
print('=== click version ===')
file_stats_click.main(['/tmp', '--recursive', '--top', '5'], standalone_mode=False)

## Practice Exercises

**Exercise 1 — CSV Filter CLI**  
Write an `argparse`-based CLI tool `csv-filter` that:
- Takes `input_csv`, `output_csv` as positional arguments
- Accepts `--column COL`, `--value VAL` to filter rows where `COL == VAL`
- Accepts `--gt FLOAT` and `--lt FLOAT` for numeric comparisons
- Prints a summary: original rows, rows after filter, rows written

**Exercise 2 — click Multi-Command Tool**  
Create a `click.group()` called `vault` with three subcommands:
- `set KEY VALUE` — store a key-value pair in `/tmp/vault.json`
- `get KEY` — retrieve and print a stored value
- `list` — show all stored keys in a rich Table

**Exercise 3 — Rich Dashboard**  
Using `rich`, create a `print_dashboard(jobs: list[dict])` function that renders:
- A title Panel with the current date and time
- A Table showing `name`, `status`, `duration`, `next_run` for each job
- A second Table with summary counts (total, running, done, failed)
Use colored status values (green=done, yellow=running, red=failed).